# 00 — Data Preparation
## IMF WEO Macroeconomic Panel | 197 Countries | 1995–2023

### Purpose
Load, reshape, clean, and prepare the raw IMF WEO dataset for analysis.

### Inputs
- `data/df_panel_raw.csv` — raw IMF WEO file

### Outputs
- `data/01_panel_cleaned.csv` — cleaned panel after interpolation and country filtering

### Dependencies
- None — this is the first notebook
- Run before: `01_feature_engineering.ipynb`


In [76]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Load raw data ─────────────────────────────────────────────
df = pd.read_csv(r"data/df_panel_raw.csv")
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (2196, 36)


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDICATOR,FREQUENCY,SCALE,1995,1996,1997,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,IMF.RES:WEO(9.0.0),GBR.BCA_NGDPD.A,OBS_VALUE,United Kingdom,"Current account balance (credit less debit), P...",Annual,Units,-0.381,-0.264,-0.140,...,-5.022,-4.948,-5.448,-3.493,-3.927,-2.688,-2.934,-0.437,-2.102,-3.509
1,IMF.RES:WEO(9.0.0),USA.GGX_NGDP.A,OBS_VALUE,United States,"Expenditure, General government, Percent of GDP",Annual,Units,NaN,NaN,NaN,...,35.324,35.031,35.333,35.194,35.349,35.819,44.722,43.201,36.790,37.650
2,IMF.RES:WEO(9.0.0),AUT.BCA_NGDPD.A,OBS_VALUE,Austria,"Current account balance (credit less debit), P...",Annual,Units,-2.876,-2.854,-2.559,...,2.397,1.577,2.574,1.264,0.846,2.376,3.367,1.735,-0.862,1.341
3,IMF.RES:WEO(9.0.0),USA.GGXWDG_NGDP.A,OBS_VALUE,United States,"Gross debt, General government, Percent of GDP",Annual,Units,NaN,NaN,NaN,...,104.930,105.424,107.380,106.374,107.625,108.751,132.513,125.009,119.104,119.836
4,IMF.RES:WEO(9.0.0),GBR.GGX_NGDP.A,OBS_VALUE,United Kingdom,"Expenditure, General government, Percent of GDP",Annual,Units,37.631,35.716,34.891,...,41.184,40.425,39.632,39.155,38.871,38.763,49.987,45.753,44.099,44.829


In [77]:
# ── Reshape wide → long → wide (pivot) ───────────────────────
df_long = df.melt(
    id_vars   = ["COUNTRY", "INDICATOR"],
    var_name  = "YEAR",
    value_name= "VALUE"
)
df_long["YEAR"] = pd.to_numeric(df_long["YEAR"], errors="coerce")
df_long = df_long.dropna(subset=["YEAR"])
df_long["YEAR"] = df_long["YEAR"].astype(int)

df_panel = df_long.pivot_table(
    index  = ["COUNTRY", "YEAR"],
    columns= "INDICATOR",
    values = "VALUE"
).reset_index()

print(f"Panel shape: {df_panel.shape}")

Panel shape: (5584, 14)


In [78]:
# ── Rename columns ────────────────────────────────────────────
df_panel.columns = df_panel.columns.str.strip()

df_panel.rename(columns={
    "All Items, Consumer price index (CPI), Period average, percent change": "Inflation",
    "Gross domestic product (GDP), Constant prices, Percent change":         "GDP_Growth",
    "Gross domestic product (GDP), Constant prices, percent change":         "GDP_Growth",
    "Unemployment rate":                                                      "Unemployment",
    "Gross debt, General government, Percent of GDP":                        "Debt",
    "Net lending (+) / net borrowing (-), General government, Percent of GDP": "Fiscal_Balance",
    "Current account balance (credit less debit), Percent of GDP":           "Current_Account",
    "Exports of goods and services, Volume, Free on board (FOB), Percent change": "Exports",
    "Exports of goods and services, volume, Free on board (FOB), Percent change": "Exports",
    "Imports of goods and services, Volume, Cost insurance freight (CIF), Percent change": "Imports",
    "Imports of goods and services, volume, Cost insurance freight (CIF), Percent change": "Imports",
    "Revenue, General government, Percent of GDP":    "Revenue",
    "Expenditure, General government, Percent of GDP":"Expenditure",
    "Gross national savings, Percent of GDP":          "Savings",
    "Gross capital formation, Percent of GDP":         "Investment",
}, inplace=True)

print("Columns:", df_panel.columns.tolist())

Columns: ['COUNTRY', 'YEAR', 'Inflation', 'Current_Account', 'Expenditure', 'Exports', 'Investment', 'Debt', 'GDP_Growth', 'Savings', 'Imports', 'Fiscal_Balance', 'Revenue', 'Unemployment']


In [79]:
# ── Basic cleaning ────────────────────────────────────────────
# Drop Unemployment (too many missing values)
if "Unemployment" in df_panel.columns:
    df_panel = df_panel.drop(columns=["Unemployment"])

# Remove duplicates and sort
df_panel = df_panel.drop_duplicates(subset=["COUNTRY","YEAR"])
df_panel = df_panel.sort_values(["COUNTRY","YEAR"]).reset_index(drop=True)

# Force numeric
for col in df_panel.columns[2:]:
    df_panel[col] = pd.to_numeric(df_panel[col], errors="coerce")

print(f"Shape after basic cleaning: {df_panel.shape}")
print(f"Countries: {df_panel['COUNTRY'].nunique()}")
print(f"Year range: {df_panel['YEAR'].min()} – {df_panel['YEAR'].max()}")

Shape after basic cleaning: (5584, 13)
Countries: 197
Year range: 1995 – 2023


In [80]:
# =============================================================================
# STEP 1 — RECORD ORIGINAL MISSING VALUES (BEFORE ANY IMPUTATION)
# =============================================================================

cols_to_interpolate = [
    "Inflation", "Current_Account", "Expenditure",
    "Exports", "Imports", "Savings", "Investment",
    "Debt", "Fiscal_Balance", "Revenue"
]

# ── Record which cells were originally missing ────────────────
# This creates a True/False matrix — True means value was missing
originally_missing = df_panel[cols_to_interpolate].isnull().copy()

# ── Summary: missing % per column ────────────────────────────
print("="*55)
print("ORIGINAL MISSING VALUES (before any imputation)")
print("="*55)
print("\nMissing % per column:")
print((df_panel[cols_to_interpolate].isnull().mean()*100)\
      .sort_values(ascending=False).round(2).to_string())

# ── Summary: missing % per country (top 20) ──────────────────
country_missing_pct = df_panel.groupby("COUNTRY")[cols_to_interpolate]\
    .apply(lambda x: x.isnull().mean().mean()) * 100

print(f"\nTotal countries     : {df_panel['COUNTRY'].nunique()}")
print(f"Countries with ANY missing value: "
      f"{(country_missing_pct > 0).sum()}")
print(f"Countries >30% missing (to be removed): "
      f"{(country_missing_pct >= 30).sum()}")
print(f"Countries >10% missing: {(country_missing_pct >= 10).sum()}")
print(f"Countries >5%  missing: {(country_missing_pct >= 5).sum()}")

print("\nTop 20 countries by missing %:")
print(country_missing_pct.sort_values(ascending=False)\
      .head(20).round(2).to_string())

# ── Total missing cells ───────────────────────────────────────
total_cells    = df_panel[cols_to_interpolate].size
missing_cells  = df_panel[cols_to_interpolate].isnull().sum().sum()
print(f"\nTotal cells in feature matrix : {total_cells}")
print(f"Originally missing cells      : {missing_cells}")
print(f"Overall missing rate          : {missing_cells/total_cells*100:.2f}%")

# ── Gap length analysis — how long are the missing runs? ──────
print("\nGap length distribution (consecutive NaN runs per country-column):")
gap_lengths = []
for col in cols_to_interpolate:
    for country, grp in df_panel.groupby("COUNTRY"):
        series = grp[col]
        mask   = series.isnull()
        if not mask.any():
            continue
        # Count consecutive NaN run lengths
        runs = mask.groupby((mask != mask.shift()).cumsum())
        for _, run in runs:
            if run.iloc[0]:   # only count NaN runs
                gap_lengths.append({
                    "Column" : col,
                    "Country": country,
                    "Gap_len": len(run)
                })

if gap_lengths:
    gap_df = pd.DataFrame(gap_lengths)
    print(f"\nTotal NaN runs found: {len(gap_df)}")
    print(f"\nGap length distribution:")
    print(gap_df["Gap_len"].value_counts().sort_index()\
          .rename("Count").to_frame().assign(
              Pct=lambda x: (x["Count"]/x["Count"].sum()*100).round(1)
          ).to_string())
    print(f"\nGaps ≤3 years (safe to interpolate): "
          f"{(gap_df['Gap_len'] <= 3).sum()} "
          f"({(gap_df['Gap_len'] <= 3).mean()*100:.1f}%)")
    print(f"Gaps >3 years (structural absence) : "
          f"{(gap_df['Gap_len'] > 3).sum()} "
          f"({(gap_df['Gap_len'] > 3).mean()*100:.1f}%)")

ORIGINAL MISSING VALUES (before any imputation)

Missing % per column:
INDICATOR
Savings            13.86
Investment         12.79
Imports            11.75
Exports            11.32
Debt                9.13
Fiscal_Balance      4.66
Expenditure         4.55
Revenue             3.87
Current_Account     3.72
Inflation           1.07

Total countries     : 197
Countries with ANY missing value: 134
Countries >30% missing (to be removed): 20
Countries >10% missing: 45
Countries >5%  missing: 68

Top 20 countries by missing %:
COUNTRY
Puerto Rico                                                        63.45
Liechtenstein, Principality of                                     60.00
Somalia                                                            54.17
Zimbabwe                                                           53.79
Nauru, Republic of                                                 50.00
Andorra, Principality of                                           48.26
San Marino, Republic of      

In [81]:
# =============================================================================
# STEP 2 — REMOVE COUNTRIES WITH >30% MISSING (ON RAW DATA)
# =============================================================================

# country_missing_pct already computed in Step 1
# Use it directly here

print("="*55)
print("STEP 2 — COUNTRY REMOVAL (>30% MISSING THRESHOLD)")
print("="*55)

# ── Identify countries to remove ─────────────────────────────
countries_to_remove = country_missing_pct[country_missing_pct >= 30]\
    .sort_values(ascending=False)
countries_to_keep   = country_missing_pct[country_missing_pct < 30].index

print(f"\nCountries to REMOVE ({len(countries_to_remove)}):")
print(countries_to_remove.round(2).to_string())

# ── Apply removal ─────────────────────────────────────────────

# ── Also update originally_missing to match ──────────────────
# Drop the removed countries from the flag matrix too
# so it stays aligned with df_panel row by row
# BEFORE the removal, save the original indices of kept countries
keep_idx = df_panel[df_panel["COUNTRY"].isin(countries_to_keep)].index

# Apply removal
df_panel = df_panel[df_panel["COUNTRY"].isin(countries_to_keep)].copy()
df_panel = df_panel.sort_values(["COUNTRY","YEAR"]).reset_index(drop=True)

# NOW align originally_missing using the saved original indices
originally_missing = originally_missing.loc[keep_idx].reset_index(drop=True)

print(f"\nCountries before removal : 197")
print(f"Countries removed        : {len(countries_to_remove)}")
print(f"Countries remaining      : {df_panel['COUNTRY'].nunique()}")
print(f"Rows remaining           : {df_panel.shape[0]}")

# ── Recheck missing % after removal ──────────────────────────
print("\nMissing % per column after country removal:")
print((df_panel[cols_to_interpolate].isnull().mean()*100)\
      .sort_values(ascending=False).round(2).to_string())

remaining_missing_cells  = df_panel[cols_to_interpolate].isnull().sum().sum()
remaining_total_cells    = df_panel[cols_to_interpolate].size
print(f"\nRemaining missing cells : {remaining_missing_cells}")
print(f"Remaining missing rate  : "
      f"{remaining_missing_cells/remaining_total_cells*100:.2f}%")

# ── Check gap distribution after removal ─────────────────────
gap_lengths_after = []
for col in cols_to_interpolate:
    for country, grp in df_panel.groupby("COUNTRY"):
        series = grp[col]
        mask   = series.isnull()
        if not mask.any():
            continue
        runs = mask.groupby((mask != mask.shift()).cumsum())
        for _, run in runs:
            if run.iloc[0]:
                gap_lengths_after.append({
                    "Column" : col,
                    "Country": country,
                    "Gap_len": len(run)
                })

if gap_lengths_after:
    gap_df2 = pd.DataFrame(gap_lengths_after)
    print(f"\nGap distribution after country removal:")
    print(f"Total NaN runs         : {len(gap_df2)}")
    print(f"Gaps ≤3 years          : "
          f"{(gap_df2['Gap_len'] <= 3).sum()} "
          f"({(gap_df2['Gap_len'] <= 3).mean()*100:.1f}%)")
    print(f"Gaps >3 years          : "
          f"{(gap_df2['Gap_len'] > 3).sum()} "
          f"({(gap_df2['Gap_len'] > 3).mean()*100:.1f}%)")

    print(f"\nGap length breakdown:")
    print(gap_df2["Gap_len"].value_counts().sort_index()\
          .rename("Count").to_frame().assign(
              Pct=lambda x: (x["Count"]/x["Count"].sum()*100).round(1)
          ).to_string())

    # Which columns still have long gaps?
    print(f"\nLong gaps (>3 years) by column:")
    print(gap_df2[gap_df2["Gap_len"] > 3]\
          .groupby("Column")["Gap_len"]\
          .agg(["count","mean","max"])\
          .round(1).sort_values("count", ascending=False).to_string())

    # Which countries have long gaps remaining?
    print(f"\nCountries with long gaps (>3 years) — top 15:")
    print(gap_df2[gap_df2["Gap_len"] > 3]\
          .groupby("Country")["Gap_len"]\
          .agg(["count","max"])\
          .sort_values("count", ascending=False)\
          .head(15).to_string())

STEP 2 — COUNTRY REMOVAL (>30% MISSING THRESHOLD)

Countries to REMOVE (20):
COUNTRY
Puerto Rico                                                        63.45
Liechtenstein, Principality of                                     60.00
Somalia                                                            54.17
Zimbabwe                                                           53.79
Nauru, Republic of                                                 50.00
Andorra, Principality of                                           48.26
San Marino, Republic of                                            48.08
Tuvalu                                                             45.65
Samoa                                                              41.03
Macao Special Administrative Region, People's Republic of China    40.87
Micronesia, Federated States of                                    40.34
Iraq                                                               38.26
Timor-Leste, Democratic Republic of    

In [82]:
print(f"Final countries : {df_panel['COUNTRY'].nunique()}")  # should be 177
print(f"Final rows      : {df_panel.shape[0]}")
print(f"Years covered   : {int(df_panel['YEAR'].min())} – {int(df_panel['YEAR'].max())}")

Final countries : 177
Final rows      : 5069
Years covered   : 1995 – 2023


In [83]:
# =============================================================================
# STEP 3 — SELECTIVE INTERPOLATION (GAPS ≤ 3 YEARS ONLY)
# =============================================================================
# Only interpolate short gaps — long gaps are left as NaN intentionally.
# This prevents inventing data across structural breaks (wars, data gaps).
 
print("="*55)
print("STEP 3 — SELECTIVE INTERPOLATION (gaps ≤ 3 years)")
print("="*55)
 
def interpolate_short_gaps(series, max_gap=3):
    """
    Linear interpolation only where the consecutive NaN run is ≤ max_gap.
    Longer runs are left as NaN.
    """
    series = series.copy()
    mask = series.isnull()
    if not mask.any():
        return series
 
    # Label each run of NaNs
    run_id = (mask != mask.shift()).cumsum()
    run_lengths = mask.groupby(run_id).transform('sum')
 
    # Only interpolate where run length ≤ max_gap
    interpolable = mask & (run_lengths <= max_gap)
    series_interp = series.copy()
    series_interp[interpolable] = np.nan   # temporarily keep these as NaN
    # Interpolate the full series, then restore long gaps
    full_interp = series.interpolate(method='linear', limit_direction='both')
    # Accept interpolated value only for short gaps
    series[interpolable] = full_interp[interpolable]
    return series
 
df_panel = df_panel.copy()
for col in cols_to_interpolate:
    df_panel[col] = (df_panel
                     .groupby("COUNTRY")[col]
                     .transform(lambda x: interpolate_short_gaps(x, max_gap=3)))
 
after_step3 = df_panel[cols_to_interpolate].isnull().sum().sum()
total_cells  = df_panel[cols_to_interpolate].size
print(f"\nMissing cells after Step 3 : {after_step3}")
print(f"Missing rate after Step 3  : {after_step3/total_cells*100:.2f}%")
 
print("\nMissing % per column after Step 3:")
print((df_panel[cols_to_interpolate].isnull().mean()*100)
      .sort_values(ascending=False)
      .round(2)
      .head(10)
      .to_string())

STEP 3 — SELECTIVE INTERPOLATION (gaps ≤ 3 years)

Missing cells after Step 3 : 1812
Missing rate after Step 3  : 3.57%

Missing % per column after Step 3:
INDICATOR
Investment         7.40
Savings            6.53
Debt               6.45
Fiscal_Balance     3.08
Expenditure        2.98
Exports            2.72
Imports            2.72
Revenue            2.45
Current_Account    1.05
Inflation          0.37


In [84]:
# STEP 4 — FFILL / BFILL WITH LIMIT = 2 (EDGE GAPS ≤ 2 YEARS)
# =============================================================================
# Handles gaps at the start/end of a country's series that interpolation
# cannot reach (no anchor point on one side).
 
print("\n" + "="*55)
print("STEP 4 — FFILL / BFILL  (limit = 2)")
print("="*55)
 
for col in cols_to_interpolate:
    df_panel[col] = (df_panel
                     .groupby("COUNTRY")[col]
                     .transform(lambda x: x.ffill(limit=2).bfill(limit=2)))
 
after_step4 = df_panel[cols_to_interpolate].isnull().sum().sum()
print(f"\nMissing cells after Step 4 : {after_step4}")
print(f"Missing rate after Step 4  : {after_step4/total_cells*100:.2f}%")
print(f"Cells filled in Step 4     : {after_step3 - after_step4}")
 


STEP 4 — FFILL / BFILL  (limit = 2)

Missing cells after Step 4 : 1492
Missing rate after Step 4  : 2.94%
Cells filled in Step 4     : 320


In [85]:
# =============================================================================
# STEP 5 — CHECK REMAINING MISSING
# =============================================================================
 
print("\n" + "="*55)
print("STEP 5 — REMAINING MISSING AFTER STEPS 3 & 4")
print("="*55)
 
remaining = (df_panel[cols_to_interpolate]
             .isnull()
             .mean()
             .mul(100)
             .round(2))
 
# Full column breakdown
print("\nMissing % per column (all cols, sorted):")
print((df_panel[cols_to_interpolate].isnull().mean()*100)
      .sort_values(ascending=False)
      .round(3)
      .to_string())
 
# Which country-column pairs still have missing?
print("\nCountry-column pairs with remaining missing values:")
still_missing = []
for col in cols_to_interpolate:
    miss_countries = (df_panel[df_panel[col].isnull()]["COUNTRY"]
                      .value_counts()
                      .reset_index())
    miss_countries.columns = ["COUNTRY", "missing_rows"]
    miss_countries["column"] = col
    still_missing.append(miss_countries)
 
if still_missing:
    miss_df = pd.concat(still_missing, ignore_index=True)
    miss_df = miss_df[miss_df["missing_rows"] > 0].sort_values(
        "missing_rows", ascending=False
    )
    print(miss_df.to_string(index=False))
    print(f"\nTotal country-column pairs still missing: {len(miss_df)}")
else:
    print("No remaining missing values ✓")
 
after_step5 = df_panel[cols_to_interpolate].isnull().sum().sum()
pct_remaining = after_step5 / total_cells * 100
print(f"\nTotal missing cells remaining : {after_step5}")
print(f"Overall missing rate          : {pct_remaining:.3f}%")
 


STEP 5 — REMAINING MISSING AFTER STEPS 3 & 4

Missing % per column (all cols, sorted):
INDICATOR
Investment         7.161
Savings            6.293
Debt               4.439
Exports            2.407
Imports            2.407
Fiscal_Balance     2.012
Expenditure        1.953
Revenue            1.539
Current_Account    0.888
Inflation          0.335

Country-column pairs with remaining missing values:
                                                            COUNTRY  missing_rows          column
                                                              Sudan            29      Investment
                                                              Libya            29            Debt
                                                       Turkmenistan            29         Savings
                                                       Turkmenistan            29      Investment
                                                             Kuwait            29         Exports
           

In [86]:
# STEP 6 — YEAR-WISE CROSS-SECTIONAL MEDIAN (RESIDUAL 1-2% GAPS ONLY)
# =============================================================================
# Last resort ONLY for the residual gaps that Steps 3 & 4 could not fill.
# This is applied sparingly — only if missing rate > 0 after Steps 3+4.
# Year-wise median is better than global median because macro conditions
# vary substantially across decades.
 
print("\n" + "="*55)
print("STEP 6 — YEAR-WISE MEDIAN (residual gaps only)")
print("="*55)
 
cells_before_step6 = df_panel[cols_to_interpolate].isnull().sum().sum()
 
if cells_before_step6 == 0:
    print("No residual missing values — Step 6 skipped ✓")
else:
    print(f"Residual missing cells entering Step 6: {cells_before_step6}")
    print(f"({cells_before_step6/total_cells*100:.3f}% of panel)")
 
    # Track exactly which cells are filled by year-wise median
    step6_filled = 0
    for col in cols_to_interpolate:
        col_missing_before = df_panel[col].isnull().sum()
        if col_missing_before == 0:
            continue
        # Fill with year-wise median (cross-sectional)
        year_medians = df_panel.groupby("YEAR")[col].transform("median")
        mask = df_panel[col].isnull()
        df_panel.loc[mask, col] = year_medians[mask]
        col_filled = col_missing_before - df_panel[col].isnull().sum()
        step6_filled += col_filled
        if col_filled > 0:
            print(f"  {col:<30s} filled {col_filled} cells via year-median")
 
    # Fallback: global median for any cells year-median couldn't fill
    # (happens if an entire year is missing for a column — very rare)
    global_filled = 0
    for col in cols_to_interpolate:
        still_null = df_panel[col].isnull().sum()
        if still_null > 0:
            global_med = df_panel[col].median()
            df_panel[col].fillna(global_med, inplace=True)
            global_filled += still_null
            print(f"  {col:<30s} filled {still_null} cells via GLOBAL median ⚠")
 
    after_step6 = df_panel[cols_to_interpolate].isnull().sum().sum()
    print(f"\nCells filled by year-median   : {step6_filled}")
    if global_filled:
        print(f"Cells filled by global median : {global_filled}  ⚠ (flag for review)")
    print(f"Missing cells after Step 6    : {after_step6}")


STEP 6 — YEAR-WISE MEDIAN (residual gaps only)
Residual missing cells entering Step 6: 1492
(2.943% of panel)
  Inflation                      filled 17 cells via year-median
  Current_Account                filled 45 cells via year-median
  Expenditure                    filled 99 cells via year-median
  Exports                        filled 122 cells via year-median
  Imports                        filled 122 cells via year-median
  Savings                        filled 319 cells via year-median
  Investment                     filled 363 cells via year-median
  Debt                           filled 225 cells via year-median
  Fiscal_Balance                 filled 102 cells via year-median
  Revenue                        filled 78 cells via year-median

Cells filled by year-median   : 1492
Missing cells after Step 6    : 0


In [87]:
# =============================================================================
# STEP 7 — DROP ROWS WHERE GDP_GROWTH IS MISSING
# =============================================================================
# GDP_Growth is the target variable. Any row without it is unusable
# for modelling and must be removed.
 
print("\n" + "="*55)
print("STEP 7 — DROP ROWS WITH MISSING GDP_GROWTH")
print("="*55)
 
rows_before = len(df_panel)
gdp_null    = df_panel["GDP_Growth"].isnull().sum()
 
df_panel = df_panel.dropna(subset=["GDP_Growth"]).reset_index(drop=True)
 
rows_after = len(df_panel)
print(f"Rows before : {rows_before}")
print(f"GDP_Growth NaNs dropped : {gdp_null}")
print(f"Rows after  : {rows_after}")
print(f"Countries retained: {df_panel['COUNTRY'].nunique()}")
print(f"Year range  : {int(df_panel['YEAR'].min())} – {int(df_panel['YEAR'].max())}")
 


STEP 7 — DROP ROWS WITH MISSING GDP_GROWTH
Rows before : 5069
GDP_Growth NaNs dropped : 17
Rows after  : 5052
Countries retained: 177
Year range  : 1995 – 2023


In [88]:
df_raw=pd.read_csv(r"data/df_panel_raw.csv")


In [89]:
# =============================================================================
# STEP 8 — ATTACH IMPUTATION FLAG COLUMNS (COMPLETE)
# =============================================================================

print("="*55)
print("STEP 8 — ATTACH IMPUTATION FLAG COLUMNS")
print("="*55)

# ── Reshape df_raw wide → long → pivot ───────────────────────
year_cols = [str(y) for y in range(1995, 2024)]

df_raw_long = df_raw[['COUNTRY', 'INDICATOR'] + year_cols].melt(
    id_vars    = ['COUNTRY', 'INDICATOR'],
    value_vars = year_cols,
    var_name   = 'YEAR',
    value_name = 'raw_value'
)
df_raw_long['YEAR'] = df_raw_long['YEAR'].astype(int)

df_raw_pivot = df_raw_long.pivot_table(
    index   = ['COUNTRY', 'YEAR'],
    columns = 'INDICATOR',
    values  = 'raw_value',
    aggfunc = 'first'
).reset_index()
df_raw_pivot.columns.name = None

# ── Rename IMF indicator names → your df_panel column names ──
INDICATOR_RENAME = {
    'Gross domestic product (GDP), Constant prices, Percent change'              : 'GDP_Growth',
    'All Items, Consumer price index (CPI), Period average, percent change'      : 'Inflation',
    'Gross debt, General government, Percent of GDP'                             : 'Debt',
    'Net lending (+) / net borrowing (-), General government, Percent of GDP'   : 'Fiscal_Balance',
    'Current account balance (credit less debit), Percent of GDP'                : 'Current_Account',
    'Exports of goods and services, Volume, Free on board (FOB), Percent change' : 'Exports',
    'Imports of goods and services, Volume, Cost insurance freight (CIF), Percent change' : 'Imports',
    'Revenue, General government, Percent of GDP'                                : 'Revenue',
    'Expenditure, General government, Percent of GDP'                            : 'Expenditure',
    'Gross national savings, Percent of GDP'                                     : 'Savings',
    'Gross capital formation, Percent of GDP'                                    : 'Investment',
    'Unemployment rate'                                                          : 'Unemployment_Rate',
}

df_raw_pivot = df_raw_pivot.rename(columns=INDICATOR_RENAME)
print(f"Renamed columns: {[c for c in df_raw_pivot.columns if c not in ['COUNTRY','YEAR']]}")

# ── Merge raw pivot against df_panel on COUNTRY + YEAR ───────
merged = df_panel[['COUNTRY', 'YEAR']].merge(
    df_raw_pivot,
    on  = ['COUNTRY', 'YEAR'],
    how = 'left'
)

# ── Create imputation flags ───────────────────────────────────
# True = value was NaN in raw data BUT is now filled in df_panel
flag_cols_list = []
for col in cols_to_interpolate:
    if col not in merged.columns:
        print(f"⚠  '{col}' not found in raw pivot — skipping flag")
        continue
    flag_name = f"{col}_imputed"
    raw_was_null = merged[col].isnull()
    now_not_null = df_panel[col].notna()
    df_panel[flag_name] = (raw_was_null & now_not_null).values
    flag_cols_list.append(flag_name)

# ── Summary ───────────────────────────────────────────────────
imputed_count = df_panel[flag_cols_list].sum().sum()
total_feat_cells = df_panel[cols_to_interpolate].size

print(f"\nImputation flags attached : {len(flag_cols_list)} columns")
print(f"Total imputed cells       : {imputed_count}")
print(f"Total feature cells       : {total_feat_cells}")
print(f"Imputation rate           : {imputed_count / total_feat_cells * 100:.2f}%")
print("\nImputed cell count per column:")
print(df_panel[flag_cols_list].sum()
      .sort_values(ascending=False)
      .to_string())

STEP 8 — ATTACH IMPUTATION FLAG COLUMNS
Renamed columns: ['Inflation', 'Current_Account', 'Expenditure', 'Exports', 'Investment', 'Debt', 'GDP_Growth', 'Savings', 'Imports', 'Fiscal_Balance', 'Revenue', 'Unemployment_Rate']

Imputation flags attached : 10 columns
Total imputed cells       : 2065
Total feature cells       : 50520
Imputation rate           : 4.09%

Imputed cell count per column:
INDICATOR
Debt_imputed               406
Investment_imputed         377
Savings_imputed            333
Fiscal_Balance_imputed     192
Expenditure_imputed        187
Exports_imputed            162
Revenue_imputed            161
Imports_imputed            158
Current_Account_imputed     59
Inflation_imputed           30


In [90]:
CLEAN_PATH = 'data/01_panel_cleaned.csv'
df_panel.to_csv(CLEAN_PATH, index=False)

print(f"Saved → {CLEAN_PATH}")
print(f"Shape  : {df_panel.shape}")
print(f"\n── Final panel summary ─────────────────────────")
print(f"  Countries   : {df_panel['COUNTRY'].nunique()}")
print(f"  Year range  : {int(df_panel['YEAR'].min())} – {int(df_panel['YEAR'].max())}")
print(f"  Rows        : {df_panel.shape[0]}")
print(f"  Columns     : {df_panel.shape[1]}")
print(f"  Missing cells (feature cols) : {df_panel[cols_to_interpolate].isnull().sum().sum()}")
print(f"  Imputation rate              : 4.09%")
print(f"\n── Imputation pipeline summary ─────────────────")
print(f"  Step 1  Raw data loaded & missing flagged")
print(f"  Step 2  20 countries removed (>30% missing)")
print(f"  Step 3  Short gaps ≤3 yrs interpolated")
print(f"  Step 4  Edge gaps ≤2 yrs ffill/bfill")
print(f"  Step 5  Residual missing checked")
print(f"  Step 6  Year-wise median for residual gaps")
print(f"  Step 7  Rows with missing GDP_Growth dropped")
print(f"  Step 8  Imputation flags attached (4.09%)")
print(f"  Step 9  Checkpoint saved ✓")
print(f"\n{'='*55}")
print(f"00_data_preparation.ipynb  COMPLETE ✓")
print(f"Next → 01_feature_engineering.ipynb")
print(f"{'='*55}")

Saved → data/01_panel_cleaned.csv
Shape  : (5052, 23)

── Final panel summary ─────────────────────────
  Countries   : 177
  Year range  : 1995 – 2023
  Rows        : 5052
  Columns     : 23
  Missing cells (feature cols) : 0
  Imputation rate              : 4.09%

── Imputation pipeline summary ─────────────────
  Step 1  Raw data loaded & missing flagged
  Step 2  20 countries removed (>30% missing)
  Step 3  Short gaps ≤3 yrs interpolated
  Step 4  Edge gaps ≤2 yrs ffill/bfill
  Step 5  Residual missing checked
  Step 6  Year-wise median for residual gaps
  Step 7  Rows with missing GDP_Growth dropped
  Step 8  Imputation flags attached (4.09%)
  Step 9  Checkpoint saved ✓

00_data_preparation.ipynb  COMPLETE ✓
Next → 01_feature_engineering.ipynb
